# Microbenchmarks on CPU
This is a notebook for microbenchmarks running on CPU.

In [1]:
# Bootstrap defaults similar to GPU notebook
import os, glob, psutil

# Ensure SPARK_HOME is set (fallback to local build dist)
SPARK_HOME = os.environ.get("SPARK_HOME", "/home/chenqh23/spark/dist")
os.environ["SPARK_HOME"] = SPARK_HOME
os.environ['JAVA_HOME'] = os.environ.get('JAVA_HOME', '/usr/lib/jvm/java-17-openjdk-amd64')

# Clean up lingering SparkSubmit JVMs that can cause empty Py4J answers
for p in psutil.process_iter(['pid','name','cmdline']):
    cl = ' '.join(p.info.get('cmdline') or [])
    if p.info.get('name') == 'java' and 'org.apache.spark.deploy.SparkSubmit' in cl:
        try:
            p.kill()
        except Exception:
            pass

# Ensure findspark can locate Spark
try:
    import findspark; findspark.init(os.environ['SPARK_HOME'])
except Exception:
    pass

# Use local mode unless a cluster URL is provided
os.environ.setdefault("SPARK_MASTER_URL", "local[*]")

# Data and event log defaults
os.environ.setdefault("DATA_ROOT", "/home/chenqh23/spark-rapids-examples/datasets")
os.environ.setdefault("EVENTLOG_DIR", "/tmp/spark-events")
try:
    os.makedirs(os.environ["EVENTLOG_DIR"], exist_ok=True)
except Exception:
    pass

# Export Intel DML paths for this notebook (driver process)
HOME = os.path.expanduser("~")
os.environ.setdefault("DML_INC", f"{HOME}/dml_install_dir/include")
os.environ.setdefault("DML_LIB", f"{HOME}/dml_install_dir/lib")

print("SPARK_HOME =", os.environ["SPARK_HOME"]) 
print("SPARK_MASTER_URL =", os.environ["SPARK_MASTER_URL"]) 
print("EVENTLOG_DIR =", os.environ["EVENTLOG_DIR"]) 
print("DML_INC =", os.environ["DML_INC"]) 
print("DML_LIB =", os.environ["DML_LIB"]) 


SPARK_HOME = /home/chenqh23/spark/dist
SPARK_MASTER_URL = local[*]
EVENTLOG_DIR = /tmp/spark-events
DML_INC = /home/chenqh23/dml_install_dir/include
DML_LIB = /home/chenqh23/dml_install_dir/lib


Run the microbenchmark with retry times

In [2]:
def runMicroBenchmark(spark, appName, query, retryTimes):
    count = 0
    total_time = 0
    # You can print the physical plan of each query
    # spark.sql(query).explain()
    while count < retryTimes:
        start = time.time()
        spark.sql(query).collect()
        end = time.time()
        total_time += round(end - start, 2)
        count = count + 1
        print("Retry times : {}, ".format(count) + appName + " microbenchmark takes {} seconds".format(round(end - start, 2)))
    print(appName + " microbenchmark takes average {} seconds after {} retries".format(round(total_time/retryTimes),retryTimes))
    with open('result.txt', 'a') as file:
        file.write("{},{},{}\n".format(appName, round(total_time/retryTimes), retryTimes))

In [3]:
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
import os, time

dataRoot = os.environ.get("DATA_ROOT", "/users/chenqh23/spark-rapids-examples/datasets")

_DSA_AGENT_JAR = "/home/chenqh23/offload/dsa-agent/out/dsa-agent-0.1.0.jar"
_DSA_NATIVE_DIR = "/home/chenqh23/offload/dsa-agent/native"

# Ensure native libs are discoverable by environment too
os.environ["LD_LIBRARY_PATH"] = f"{_DSA_NATIVE_DIR}:{os.environ.get('DML_LIB','')}:{os.environ.get('QPL_LIB','')}:{os.environ.get('LD_LIBRARY_PATH','')}"

_DEF_OPENS = (
    "--add-opens=java.base/java.lang=ALL-UNNAMED "
    "--add-opens=java.base/java.lang.invoke=ALL-UNNAMED "
    "--add-opens=java.base/java.lang.reflect=ALL-UNNAMED "
    "--add-opens=java.base/java.io=ALL-UNNAMED "
    "--add-opens=java.base/java.net=ALL-UNNAMED "
    "--add-opens=java.base/java.nio=ALL-UNNAMED "
    "--add-opens=java.base/java.util=ALL-UNNAMED "
    "--add-opens=java.base/java.util.concurrent=ALL-UNNAMED "
    "--add-opens=java.base/jdk.internal.ref=ALL-UNNAMED "
    "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED "
)

base = (SparkConf()
    .setMaster(os.environ.get("SPARK_MASTER_URL", "local[*]"))
    .setAppName("Microbenchmark (Accel)")
    .set("spark.driver.memory", os.environ.get("DRIVER_MEM", "12g"))
    .set("spark.sql.adaptive.enabled", "true")
    .set("spark.sql.files.maxPartitionBytes", os.environ.get("MAX_PARTITION_BYTES", "128m"))
    .set("spark.sql.shuffle.partitions", os.environ.get("SHUFFLE_PARTITIONS", "96"))
    .set("spark.locality.wait", "0")
    .set("spark.scheduler.mode", "FAIR")
    .set("spark.eventLog.enabled", "false")
    .set("spark.executorEnv.DML_INC", os.environ["DML_INC"])
    .set("spark.executorEnv.DML_LIB", os.environ["DML_LIB"])
)

use_agent = os.path.exists(_DSA_AGENT_JAR) and os.path.exists(os.path.join(_DSA_NATIVE_DIR, "libdsa_copy.so"))
if use_agent:
    base = (base
        # Ensure agent classes on classpath/boot for driver and executors
        .set("spark.driver.extraClassPath", _DSA_AGENT_JAR)
        .set("spark.executor.extraClassPath", _DSA_AGENT_JAR)
        .set("spark.driver.extraJavaOptions",
             f"{_DEF_OPENS} -Xbootclasspath/a:{_DSA_AGENT_JAR} "
             f"-javaagent:{_DSA_AGENT_JAR} -Djava.library.path={_DSA_NATIVE_DIR} -Ddsa.skip.retransform=true")
        .set("spark.executor.extraJavaOptions",
             f"{_DEF_OPENS} -Xbootclasspath/a:{_DSA_AGENT_JAR} "
             f"-javaagent:{_DSA_AGENT_JAR} -Djava.library.path={_DSA_NATIVE_DIR} -Ddsa.skip.retransform=true")
        # Native libs for driver/executors
        .set("spark.driver.extraLibraryPath",
             f"{_DSA_NATIVE_DIR}:{os.environ['DML_LIB']}:{os.environ.get('QPL_LIB','')}")
        .set("spark.executor.extraLibraryPath",
             f"{_DSA_NATIVE_DIR}:{os.environ['DML_LIB']}:{os.environ.get('QPL_LIB','')}")
        # Also pass env vars to executors
        .set("spark.executorEnv.DML_INC", os.environ["DML_INC"])
        .set("spark.executorEnv.DML_LIB", os.environ["DML_LIB"])
        .set("spark.executorEnv.QPL_LIB", os.environ.get("QPL_LIB",""))
        .set("spark.executorEnv.LD_LIBRARY_PATH",
             f"{_DSA_NATIVE_DIR}:{os.environ['DML_LIB']}:{os.environ.get('QPL_LIB','')}:${{LD_LIBRARY_PATH}}")
    )
else:
    print("DSA agent not enabled: missing JAR or native lib",
          os.path.exists(_DSA_AGENT_JAR), os.path.exists(os.path.join(_DSA_NATIVE_DIR, 'libdsa_copy.so')))

# Optional: one-shot test to rule out agent-caused crash
# base = base.set("spark.driver.extraJavaOptions", f"{_DEF_OPENS} -Ddsa.disable=true") \
#            .set("spark.executor.extraJavaOptions", f"{_DEF_OPENS} -Ddsa.disable=true")

spark = SparkSession.builder.config(conf=base).getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/17 19:10:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
# DSA sanity check (bootstrap-load the JNI, then verify)
jvm = spark._jvm
print("java.library.path:", jvm.java.lang.System.getProperty("java.library.path"))
print("LD_LIBRARY_PATH:", jvm.java.lang.System.getenv("LD_LIBRARY_PATH"))

Array = jvm.java.lang.reflect.Array
ByteTYPE = jvm.java.lang.Byte.TYPE

n = 10_000_000
src = Array.newInstance(ByteTYPE, n)
dst = Array.newInstance(ByteTYPE, n)

Dsa = jvm.com.example.dsa.DsaArrayCopy
# Ensure the bootstrap classloader loads the JNI (fallback to absolute path if needed)
jvm.java.lang.System.setProperty("dsa.native.path", "/home/chenqh23/offload/dsa-agent/native/libdsa_copy.so")
print("ensureNativeLoaded:", Dsa.ensureNativeLoaded())

try:
    print("isDmlBuilt:", Dsa.isDmlBuilt())
except Exception as e:
    print("isDmlBuilt call failed:", e)

print("backend_before:", Dsa.getLastBackend())
Dsa.arraycopy(src, 0, dst, 0, n)
print("backend_after:", Dsa.getLastBackend(), "dmlStatus:", Dsa.getLastDmlStatus())
print("Note: backend=3=DML-HL, 2=DML-LL, 1=memmove-after-DML-fail, 0=memmove")

java.library.path: /home/chenqh23/offload/dsa-agent/native
LD_LIBRARY_PATH: /home/chenqh23/offload/dsa-agent/native:/home/chenqh23/dml_install_dir/lib:::/home/chenqh23/offload/dsa-agent/native:/home/chenqh23/dml_install_dir/lib:
ensureNativeLoaded: True
isDmlBuilt: True
backend_before: 0
backend_after: 3 dmlStatus: 0
Note: backend=3=DML-HL, 2=DML-LL, 1=memmove-after-DML-fail, 0=memmove


In [5]:
# Load dataframe and create tempView (avoid extreme repartition)
spark.read.parquet(dataRoot + "/tpcds/customer").createOrReplaceTempView("customer")
spark.read.parquet(dataRoot + "/tpcds/store_sales").createOrReplaceTempView("store_sales")
spark.read.parquet(dataRoot + "/tpcds/catalog_sales").createOrReplaceTempView("catalog_sales")
spark.read.parquet(dataRoot + "/tpcds/web_sales").createOrReplaceTempView("web_sales")
spark.read.parquet(dataRoot + "/tpcds/item").createOrReplaceTempView("item")
spark.read.parquet(dataRoot + "/tpcds/date_dim").createOrReplaceTempView("date_dim")

# Cache hot tables to reduce I/O contention during parallel runs
for t in ("customer","store_sales","catalog_sales","web_sales","item","date_dim"):
    try:
        spark.catalog.cacheTable(t)
    except Exception:
        pass
# Materialize caches once to warm up
for t in ("customer","store_sales","catalog_sales","web_sales","item","date_dim"):
    _ = spark.table(t).count()

print("-"*50)
time.sleep(2)

25/11/17 19:11:07 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


--------------------------------------------------


### Expand&HashAggregate
This is a microbenchmark about Expand&HashAggregate expressions running on the CPU. The query calculates the distinct value of some dimension columns and average birth year by different c_salutation of customers after grouping by c_current_hdemo_sk.

In [6]:
query0 = '''
select c_current_hdemo_sk,
count(DISTINCT if(c_salutation=="Ms.",c_salutation,null)) as c1,
count(DISTINCT if(c_salutation=="Mr.",c_salutation,null)) as c12,
count(DISTINCT if(c_salutation=="Dr.",c_salutation,null)) as c13,

count(DISTINCT if(c_salutation=="Ms.",c_first_name,null)) as c2,
count(DISTINCT if(c_salutation=="Mr.",c_first_name,null)) as c22,
count(DISTINCT if(c_salutation=="Dr.",c_first_name,null)) as c23,

count(DISTINCT if(c_salutation=="Ms.",c_last_name,null)) as c3,
count(DISTINCT if(c_salutation=="Mr.",c_last_name,null)) as c32,
count(DISTINCT if(c_salutation=="Dr.",c_last_name,null)) as c33,

count(DISTINCT if(c_salutation=="Ms.",c_birth_country,null)) as c4,
count(DISTINCT if(c_salutation=="Mr.",c_birth_country,null)) as c42,
count(DISTINCT if(c_salutation=="Dr.",c_birth_country,null)) as c43,

count(DISTINCT if(c_salutation=="Ms.",c_email_address,null)) as c5,
count(DISTINCT if(c_salutation=="Mr.",c_email_address,null)) as c52,
count(DISTINCT if(c_salutation=="Dr.",c_email_address,null)) as c53,

count(DISTINCT if(c_salutation=="Ms.",c_login,null)) as c6,
count(DISTINCT if(c_salutation=="Mr.",c_login,null)) as c62,
count(DISTINCT if(c_salutation=="Dr.",c_login,null)) as c63,

count(DISTINCT if(c_salutation=="Ms.",c_preferred_cust_flag,null)) as c7,
count(DISTINCT if(c_salutation=="Mr.",c_preferred_cust_flag,null)) as c72,
count(DISTINCT if(c_salutation=="Dr.",c_preferred_cust_flag,null)) as c73,

count(DISTINCT if(c_salutation=="Ms.",c_birth_month,null)) as c8,
count(DISTINCT if(c_salutation=="Mr.",c_birth_month,null)) as c82,
count(DISTINCT if(c_salutation=="Dr.",c_birth_month,null)) as c83,

avg(if(c_salutation=="Ms.",c_birth_year,null)) as avg1,
avg(if(c_salutation=="Mr.",c_birth_year,null)) as avg2,
avg(if(c_salutation=="Dr.",c_birth_year,null)) as avg3,
avg(if(c_salutation=="Miss.",c_birth_year,null)) as avg4,
avg(if(c_salutation=="Mrs.",c_birth_year,null)) as avg5,
avg(if(c_salutation=="Sir.",c_birth_year,null)) as avg6,
avg(if(c_salutation=="Professor.",c_birth_year,null)) as avg7,
avg(if(c_salutation=="Teacher.",c_birth_year,null)) as avg8,
avg(if(c_salutation=="Agent.",c_birth_year,null)) as avg9,
avg(if(c_salutation=="Director.",c_birth_year,null)) as avg10
from customer group by c_current_hdemo_sk
'''
print("-"*50)

--------------------------------------------------


In [7]:
# Run microbenchmark with n retry time
runMicroBenchmark(spark,"Expand&HashAggregate", query0, 2)
time.sleep(2)

Retry times : 1, Expand&HashAggregate microbenchmark takes 9.17 seconds


Retry times : 2, Expand&HashAggregate microbenchmark takes 8.24 seconds
Expand&HashAggregate microbenchmark takes average 9 seconds after 2 retries


### Windowing (without data skew)
This is a microbenchmark about windowing expressions running on CPU mode. The sub-query calculates the average ss_sales_price of a fixed window function partition by ss_customer_sk, and the parent query calculates the average price of the sub-query grouping by each customer.

In [8]:
query1 = '''
select ss_customer_sk,avg(avg_price) as avg_price
from
(
SELECT ss_customer_sk ,avg(ss_sales_price) OVER (PARTITION BY ss_customer_sk order by ss_sold_date_sk ROWS BETWEEN 50 PRECEDING AND 50 FOLLOWING ) as avg_price
FROM store_sales
where ss_customer_sk is not null
) group by ss_customer_sk order by 2 desc 
'''
print("-"*50)

--------------------------------------------------


In [9]:
# Run microbenchmark with n retry time
runMicroBenchmark(spark,"Windowing without skew", query1, 2)
time.sleep(2)

Retry times : 1, Windowing without skew microbenchmark takes 9.16 seconds


Retry times : 2, Windowing without skew microbenchmark takes 5.47 seconds
Windowing without skew microbenchmark takes average 7 seconds after 2 retries


### Windowing(with data skew)
Data skew is caused by many null values in the ss_customer_sk column.

In [10]:
query2 = '''
select ss_customer_sk,avg(avg_price) as avg_price
from
(
SELECT ss_customer_sk ,avg(ss_sales_price) OVER (PARTITION BY ss_customer_sk order by ss_sold_date_sk ROWS BETWEEN 50 PRECEDING AND 50 FOLLOWING ) as avg_price
FROM store_sales
) group by ss_customer_sk order by 2 desc 
'''
print("-"*50)

--------------------------------------------------


In [11]:
# Run microbenchmark with n retry time
runMicroBenchmark(spark,"Windowing with skew", query2, 2)
time.sleep(2)

Retry times : 1, Windowing with skew microbenchmark takes 10.85 seconds


25/11/17 19:12:32 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


Retry times : 2, Windowing with skew microbenchmark takes 9.72 seconds
Windowing with skew microbenchmark takes average 10 seconds after 2 retries


### Intersection
This is a microbenchmark about intersection operation running on CPU mode. The query calculates items in the same brand, class, and category that are sold in all three sales channels in two consecutive years.

In [12]:
query3 = '''
select i_item_sk ss_item_sk
 from item,
    (select iss.i_brand_id brand_id, iss.i_class_id class_id, iss.i_category_id category_id
     from store_sales, item iss, date_dim d1
     where ss_item_sk = iss.i_item_sk
                    and ss_sold_date_sk = d1.d_date_sk
       and d1.d_year between 1999 AND 1999 + 2
   intersect
     select ics.i_brand_id, ics.i_class_id, ics.i_category_id
     from catalog_sales, item ics, date_dim d2
     where cs_item_sk = ics.i_item_sk
       and cs_sold_date_sk = d2.d_date_sk
       and d2.d_year between 1999 AND 1999 + 2
   intersect
     select iws.i_brand_id, iws.i_class_id, iws.i_category_id
     from web_sales, item iws, date_dim d3
     where ws_item_sk = iws.i_item_sk
       and ws_sold_date_sk = d3.d_date_sk
       and d3.d_year between 1999 AND 1999 + 2) x
 where i_brand_id = brand_id
   and i_class_id = class_id
   and i_category_id = category_id
'''

In [13]:
# Run microbenchmark with n retry time
runMicroBenchmark(spark,"NDS Q14a subquery", query3, 2)
time.sleep(2)

Retry times : 1, NDS Q14a subquery microbenchmark takes 4.96 seconds


Retry times : 2, NDS Q14a subquery microbenchmark takes 3.07 seconds
NDS Q14a subquery microbenchmark takes average 4 seconds after 2 retries


In [14]:
# Run the 4 micro-benchmarks concurrently on one SparkSession/CPU
# Requires: query0, query1, query2, query3 already defined; temp views already created.

from concurrent.futures import ThreadPoolExecutor, as_completed
import time, os

# Tune quickly if you hit contention
spark.conf.set("spark.sql.files.maxPartitionBytes", os.environ.get("MB_MAX_PART_BYTES", "128m"))

def run_queries_in_pool(pool_name: str, query, retryTimes: int = 1):
    count = 0
    sc = spark.sparkContext
    t0 = time.time()
    sc.setLocalProperty("spark.scheduler.pool", pool_name)  # assign this job to a pool
    sc.setJobGroup(f"{pool_name}", f"{pool_name} run", True)
    while count < retryTimes:
        print(f"run {count+1} of {retryTimes} for {pool_name}")
        spark.sql(query).collect()  # blocking action submits a job
        count += 1
    return pool_name, round(time.time() - t0, 1)

jobs = [
    ("poolA", query0),
    ("poolB", query1),
    ("poolC", query2),
    ("poolD", query3),
    ("poolE", query0),
    ("poolF", query1),
    ("poolG", query2),
    ("poolH", query3),
    
]

# You can tune the parallelism here quickly if you hit contention
PARALLEL_JOBS = int(os.environ.get("MB_PARALLEL_JOBS", "8"))

with ThreadPoolExecutor(max_workers=PARALLEL_JOBS) as ex:
    futs = [ex.submit(run_queries_in_pool, p, query) for p, query in jobs[:PARALLEL_JOBS]]
    for f in as_completed(futs):
        name, secs = f.result()
        print(f"{name} took {secs}s")

run 1 of 1 for poolA
run 1 of 1 for poolB
run 1 of 1 for poolC
run 1 of 1 for poolD
run 1 of 1 for poolF
run 1 of 1 for poolE
run 1 of 1 for poolG
run 1 of 1 for poolH


25/11/17 19:12:53 WARN FairSchedulableBuilder: A job was submitted with scheduler pool poolC, which has not been configured. This can happen when the file that pools are read from isn't set, or when that file doesn't contain poolC. Created poolC with default configuration (schedulingMode: FIFO, minShare: 0, weight: 1)
25/11/17 19:12:53 WARN FairSchedulableBuilder: A job was submitted with scheduler pool poolG, which has not been configured. This can happen when the file that pools are read from isn't set, or when that file doesn't contain poolG. Created poolG with default configuration (schedulingMode: FIFO, minShare: 0, weight: 1)
25/11/17 19:12:53 WARN FairSchedulableBuilder: A job was submitted with scheduler pool poolF, which has not been configured. This can happen when the file that pools are read from isn't set, or when that file doesn't contain poolF. Created poolF with default configuration (schedulingMode: FIFO, minShare: 0, weight: 1)
25/11/17 19:12:53 WARN FairSchedulableBu

25/11/17 19:12:53 WARN FairSchedulableBuilder: A job was submitted with scheduler pool poolD, which has not been configured. This can happen when the file that pools are read from isn't set, or when that file doesn't contain poolD. Created poolD with default configuration (schedulingMode: FIFO, minShare: 0, weight: 1)
25/11/17 19:12:54 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/11/17 19:12:54 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/11/17 19:12:54 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/11/17 19:12:54 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/11/17 19:12:56 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/11/17 19:12:56 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will no

poolH took 15.5s
poolD took 15.7s


25/11/17 19:13:09 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/11/17 19:13:09 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/11/17 19:13:11 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/11/17 19:13:11 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


poolB took 20.1s
poolF took 20.3s


poolC took 24.0s
poolG took 25.0s


poolA took 28.1s
poolE took 28.8s
